# Benchmark & Load Testing: Hệ thống Quản lý Kho hàng, Khách hàng và Đơn hàng (Spring Boot 4 + MySQL 8)

Kịch bản test tải và đo đạc thông số hệ thống (CPU, RAM, RPS, Latency) trên Kaggle CPU:
- Setup môi trường: OpenJDK 21, MySQL Server 8.0, Maven, Locust, psutil.
- Khởi động MySQL local và khởi tạo database `KTPM`.
- Build backend bằng Maven wrapper và chạy background process (Flyway tự động migrate bảng).
- Seed dữ liệu qua API: 1 tài khoản admin, 1 khách hàng, 1 sản phẩm "hot" tồn kho thấp và 7 sản phẩm tồn kho lớn.
- Chạy Locust mô phỏng luồng người dùng: đăng nhập, tra cứu, đặt đơn tranh chấp sản phẩm hot (chống overselling), đặt đơn nhiều sản phẩm với thứ tự xáo trộn (chống deadlock), hủy đơn hoàn kho.
- Kiểm tra tính nhất quán tồn kho sau khi test: `tồn kho còn lại + số lượng đã bán = tồn kho ban đầu` và tồn kho không âm.
- Xuất bảng thống kê và vẽ biểu đồ.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq openjdk-21-jdk mysql-server maven curl psmisc > /dev/null 2>&1
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java > /dev/null 2>&1
!update-alternatives --set javac /usr/lib/jvm/java-21-openjdk-amd64/bin/javac > /dev/null 2>&1
!pip install -q locust psutil requests pandas matplotlib pymysql

print("Java version:")
!java -version
print("\nMySQL version:")
!mysql --version

In [ ]:
import os
import shutil

repo_url = "https://github.com/24020196/BTL-KTPM.git"
project_dir = "/kaggle/working/project"

if os.path.exists(project_dir):
    shutil.rmtree(project_dir)

input_paths = [os.path.join(dp, f) for dp, dn, fn in os.walk("/kaggle/input") for f in fn if f == "pom.xml"]
if input_paths:
    src_folder = os.path.dirname(input_paths[0])
    shutil.copytree(src_folder, project_dir)
    print(f"Copied source code from Kaggle Input: {src_folder}")
else:
    !git clone {repo_url} {project_dir}

pom_dirs = [dp for dp, dn, fn in os.walk(project_dir) for f in fn if f == "pom.xml"]
backend_dir = pom_dirs[0] if pom_dirs else project_dir

os.chdir(backend_dir)
print("Working directory:", os.getcwd())
!ls -la

In [ ]:
import subprocess
import time

print("Starting MySQL service...")
subprocess.run(["service", "mysql", "start"], check=True)
time.sleep(2)

init_sql = """
DROP DATABASE IF EXISTS KTPM;
CREATE DATABASE IF NOT EXISTS KTPM CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
CREATE USER IF NOT EXISTS 'ktpm_app'@'localhost' IDENTIFIED BY '12345678';
GRANT ALL PRIVILEGES ON KTPM.* TO 'ktpm_app'@'localhost';
GRANT ALL PRIVILEGES ON KTPM.* TO 'root'@'localhost';
FLUSH PRIVILEGES;
"""

res = subprocess.run(["mysql", "-u", "root", "-e", init_sql], capture_output=True, text=True)
if res.returncode != 0:
    subprocess.run(["mysql", "-u", "root", "-p12345678", "-e", init_sql], capture_output=True, text=True)

subprocess.run([
    "mysql", "-u", "root", "-e",
    "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY '12345678'; FLUSH PRIVILEGES;"
], capture_output=True)

print("MySQL initialized: database KTPM ready.")

In [ ]:
import os

env_content = """DB_NAME=KTPM
DB_USERNAME=root
DB_PASSWORD=12345678
SPRING_DATASOURCE_URL=jdbc:mysql://localhost:3306/KTPM?createDatabaseIfNotExist=true&useSSL=false&serverTimezone=UTC&allowPublicKeyRetrieval=true
SPRING_DATASOURCE_USERNAME=root
SPRING_DATASOURCE_PASSWORD=12345678
SERVER_PORT=3000
JWT_SECRET=404E635266556A586E3272357538782F413F4428472B4B6250645367566B5970
JWT_EXPIRATION=86400000
"""

env_file_path = os.path.join(backend_dir, ".env")
with open(env_file_path, "w", encoding="utf-8") as f:
    f.write(env_content)

print("Created config file:", env_file_path)

In [ ]:
import os
import subprocess

os.chdir(backend_dir)
!sed -i 's/\r$//' mvnw 2>/dev/null || true
!chmod +x mvnw

print("Building Spring Boot JAR...")
build_process = subprocess.run(["./mvnw", "clean", "package", "-DskipTests", "-B"], capture_output=True, text=True)

if build_process.returncode == 0:
    jars = [f for f in os.listdir("target") if f.endswith(".jar") and not f.endswith(".original") and not f.endswith("-sources.jar")]
    print(f"Build success! JAR: target/{jars[0]}")
else:
    print("Build failed! Log:")
    print(build_process.stdout[-2000:])
    print(build_process.stderr[-1000:])

In [ ]:
locust_code = """import json
import random
import uuid
from locust import HttpUser, task, between, SequentialTaskSet

with open("/kaggle/working/bench_config.json", encoding="utf-8") as f:
    CONFIG = json.load(f)

CUSTOMER = CONFIG["customer_number"]
HOT_PRODUCT = CONFIG["hot_product"]
MULTI_PRODUCTS = CONFIG["multi_products"]


def is_out_of_stock(res):
    return res.status_code == 400 and "tồn kho" in res.text


class WarehouseCustomerJourney(SequentialTaskSet):
    def on_start(self):
        self.product_id = HOT_PRODUCT
        self.order_id = None

    @task
    def browse_products(self):
        with self.client.get("/api/v1/products?page=0&size=10", headers=self.user.auth_headers,
                             name="01_GET_products", catch_response=True) as res:
            if res.status_code != 200:
                res.failure(f"{res.status_code}: {res.text[:200]}")
                return
            items = res.json().get("content", [])
            if items:
                self.product_id = random.choice(items)["id"]

    @task
    def view_product_detail(self):
        with self.client.get(f"/api/v1/products/{self.product_id}", headers=self.user.auth_headers,
                             name="02_GET_product_detail", catch_response=True) as res:
            if res.status_code != 200:
                res.failure(f"{res.status_code}: {res.text[:200]}")

    @task
    def browse_customers(self):
        with self.client.get("/api/v1/customers?page=0&size=10", headers=self.user.auth_headers,
                             name="03_GET_customers", catch_response=True) as res:
            if res.status_code != 200:
                res.failure(f"{res.status_code}: {res.text[:200]}")

    @task
    def browse_orders(self):
        with self.client.get("/api/v1/orders?page=0&size=10", headers=self.user.auth_headers,
                             name="04_GET_orders", catch_response=True) as res:
            if res.status_code != 200:
                res.failure(f"{res.status_code}: {res.text[:200]}")

    @task
    def create_order_hot_product(self):
        if random.random() >= 0.65:
            return
        payload = {
            "customerNumber": CUSTOMER,
            "items": [{"productCode": HOT_PRODUCT, "quantityOrdered": random.randint(1, 3)}],
        }
        with self.client.post("/api/v1/orders", json=payload, headers=self.user.auth_headers,
                              name="05_POST_orders_hot_product", catch_response=True) as res:
            if res.status_code == 201:
                self.order_id = res.json()["orderNumber"]
            elif is_out_of_stock(res):
                res.request_meta["name"] = "05_POST_orders_hot_product_out_of_stock"
                res.success()
            else:
                res.failure(f"{res.status_code}: {res.text[:200]}")

    @task
    def create_order_multi_product(self):
        if random.random() >= 0.40:
            return
        items = [{"productCode": code, "quantityOrdered": 1} for code in MULTI_PRODUCTS]
        random.shuffle(items)
        payload = {"customerNumber": CUSTOMER, "items": items}
        with self.client.post("/api/v1/orders", json=payload, headers=self.user.auth_headers,
                              name="06_POST_orders_multi_product", catch_response=True) as res:
            if res.status_code != 201:
                res.failure(f"{res.status_code}: {res.text[:200]}")

    @task
    def view_order_detail(self):
        if not self.order_id:
            return
        with self.client.get(f"/api/v1/orders/{self.order_id}", headers=self.user.auth_headers,
                             name="07_GET_order_detail", catch_response=True) as res:
            if res.status_code != 200:
                res.failure(f"{res.status_code}: {res.text[:200]}")

    @task
    def cancel_order(self):
        if self.order_id and random.random() < 0.20:
            with self.client.patch(f"/api/v1/orders/{self.order_id}", json={"status": "CANCELLED"},
                                   headers=self.user.auth_headers,
                                   name="08_PATCH_order_cancel", catch_response=True) as res:
                if res.status_code != 200:
                    res.failure(f"{res.status_code}: {res.text[:200]}")
        self.interrupt()


class WarehouseLoadUser(HttpUser):
    wait_time = between(1.0, 2.5)
    tasks = {WarehouseCustomerJourney: 5}

    def on_start(self):
        with open("/kaggle/working/jwt_token.txt", encoding="utf-8") as f:
            token = f.read().strip()

        if random.random() < 0.70:
            credentials = {"username": f"user_{uuid.uuid4().hex[:8]}", "password": "Password123!"}
            with self.client.post("/api/v1/auth/register", json=credentials,
                                  name="00_POST_register", catch_response=True) as res:
                if res.status_code != 201:
                    res.failure(f"{res.status_code}: {res.text[:200]}")
            with self.client.post("/api/v1/auth/login", json=credentials,
                                  name="00_POST_login", catch_response=True) as res:
                if res.status_code == 200:
                    token = res.json()["accessToken"]
                else:
                    res.failure(f"{res.status_code}: {res.text[:200]}")

        self.auth_headers = {"Authorization": f"Bearer {token}"}

    @task(1)
    def casual_swagger(self):
        target = random.choice(["/v3/api-docs", "/swagger-ui/index.html"])
        self.client.get(target, name=f"casual_GET_{target}")
"""

with open("/kaggle/working/locustfile.py", "w", encoding="utf-8") as f:
    f.write(locust_code)

print("Generated /kaggle/working/locustfile.py")

In [ ]:
import json
import subprocess
import time
import requests
import os
import pymysql

subprocess.run(["fuser", "-k", "3000/tcp"], capture_output=True)
time.sleep(1.5)

env = os.environ.copy()
env["DB_NAME"] = "KTPM"
env["DB_USERNAME"] = "root"
env["DB_PASSWORD"] = "12345678"
env["SPRING_DATASOURCE_URL"] = "jdbc:mysql://localhost:3306/KTPM?createDatabaseIfNotExist=true&useSSL=false&serverTimezone=UTC&allowPublicKeyRetrieval=true"
env["SPRING_DATASOURCE_USERNAME"] = "root"
env["SPRING_DATASOURCE_PASSWORD"] = "12345678"
env["SERVER_PORT"] = "3000"
env["JWT_SECRET"] = "404E635266556A586E3272357538782F413F4428472B4B6250645367566B5970"
env["JWT_EXPIRATION"] = "86400000"

jars = [f for f in os.listdir(f"{backend_dir}/target") if f.endswith(".jar") and not f.endswith(".original") and not f.endswith("-sources.jar")]
jar_path = os.path.join(backend_dir, "target", jars[0])

log_file = open("/kaggle/working/spring_boot_server.log", "w")
server_proc = subprocess.Popen(
    ["java", "-Xms512m", "-Xmx1536m", "-jar", jar_path],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
    cwd=backend_dir,
    text=True
)

print(f"Starting Spring Boot (PID: {server_proc.pid}). Waiting for health check...")

server_up = False
for attempt in range(45):
    try:
        r = requests.get("http://127.0.0.1:3000/swagger-ui/index.html", timeout=2)
        if r.status_code == 200:
            server_up = True
            print(f"Server is up and running (took ~{attempt*2}s).")
            break
    except requests.exceptions.RequestException:
        time.sleep(2)

if not server_up:
    print("Server failed to start. Last log lines:")
    with open("/kaggle/working/spring_boot_server.log", "r") as f:
        print("".join(f.readlines()[-40:]))
else:
    print("Seeding sample data via API & MySQL...")
    base_url = "http://127.0.0.1:3000/api/v1"
    admin = {"username": "admin_bench", "password": "AdminPassword123!"}
    HOT_STOCK = 300
    BULK_STOCK = 50000

    requests.post(f"{base_url}/auth/register", json=admin, timeout=5).raise_for_status()

    conn = pymysql.connect(host="localhost", user="root", password="12345678", database="KTPM")
    with conn.cursor() as cur:
        cur.execute("UPDATE users SET role = 'ROLE_ADMIN' WHERE username = %s", (admin["username"],))
    conn.commit()
    conn.close()

    login_res = requests.post(f"{base_url}/auth/login", json=admin, timeout=5)
    login_res.raise_for_status()
    token = login_res.json()["accessToken"]
    with open("/kaggle/working/jwt_token.txt", "w", encoding="utf-8") as f:
        f.write(token)
    admin_headers = {"Authorization": f"Bearer {token}"}

    customer_res = requests.post(f"{base_url}/customers", json={
        "customerName": "Khach Hang Benchmark", "phone": "0912345678", "address": "Ha Noi"
    }, headers=admin_headers, timeout=5)
    customer_res.raise_for_status()
    customer_number = customer_res.json()["customerNumber"]

    product_ids = []
    for i in range(1, 9):
        p_res = requests.post(f"{base_url}/products", json={
            "name": f"Product Benchmark {i}",
            "qty": HOT_STOCK if i == 1 else BULK_STOCK,
            "price": 100000 * i
        }, headers=admin_headers, timeout=5)
        p_res.raise_for_status()
        product_ids.append(p_res.json()["id"])

    bench_config = {
        "customer_number": customer_number,
        "hot_product": product_ids[0],
        "multi_products": product_ids[1:3],
        "initial_stock": {str(pid): (HOT_STOCK if idx == 0 else BULK_STOCK) for idx, pid in enumerate(product_ids)},
    }
    with open("/kaggle/working/bench_config.json", "w", encoding="utf-8") as f:
        json.dump(bench_config, f, indent=2)

    print(f"Seeded admin, customer {customer_number}, products {product_ids} (hot product {product_ids[0]}: {HOT_STOCK} in stock).")

In [ ]:
# Monitor system resources va run Locust load test
import threading
import psutil
import time
import subprocess

backend_host = "http://127.0.0.1:3000"

system_metrics = []
stop_monitoring = threading.Event()

def monitor_resources(interval=1.0):
    psutil.cpu_percent(interval=None)
    while not stop_monitoring.is_set():
        cpu_total = psutil.cpu_percent(interval=interval)
        cpu_cores = psutil.cpu_percent(interval=None, percpu=True)
        mem = psutil.virtual_memory()
        system_metrics.append({
            "timestamp": time.time(),
            "cpu_total_pct": cpu_total,
            "cpu_cores": cpu_cores,
            "ram_used_mb": mem.used / (1024 * 1024),
            "ram_pct": mem.percent
        })

monitor_thread = threading.Thread(target=monitor_resources, args=(1.0,), daemon=True)
monitor_thread.start()

# Load test parameters
users = 80
spawn_rate = 4
duration = "2m"

print(f"Running Locust ({users} users, {spawn_rate}/s, duration {duration})...")
locust_cmd = [
    "locust",
    "-f", "/kaggle/working/locustfile.py",
    "--headless",
    "--host", backend_host,
    "-u", str(users),
    "-r", str(spawn_rate),
    "-t", duration,
    "--csv", "/kaggle/working/kaggle_load_results",
    "--csv-full-history",
    "--html", "/kaggle/working/kaggle_load_report.html",
    "--only-summary"
]

locust_run = subprocess.run(locust_cmd, capture_output=True, text=True)
print("\nLocust summary output:")
print((locust_run.stdout + locust_run.stderr)[-3000:])

stop_monitoring.set()
monitor_thread.join(timeout=3)
server_proc.terminate()
server_proc.wait(timeout=5)
print("Load test finished. Server stopped.")

In [ ]:
# Kiem tra tinh nhat quan ton kho sau load test
import json
import pymysql

with open("/kaggle/working/bench_config.json", encoding="utf-8") as f:
    bench_config = json.load(f)
initial_stock = {int(k): v for k, v in bench_config["initial_stock"].items()}
checked = [bench_config["hot_product"], *bench_config["multi_products"]]
placeholders = ", ".join(["%s"] * len(checked))

conn = pymysql.connect(host="localhost", user="root", password="12345678", database="KTPM")
with conn.cursor() as cur:
    cur.execute(f"SELECT product_code, quantity_in_stock FROM products WHERE product_code IN ({placeholders})", checked)
    stock = dict(cur.fetchall())
    cur.execute(f"""
        SELECT d.product_code, SUM(d.quantity_ordered)
        FROM order_details d
        JOIN orders o ON o.order_number = d.order_number
        WHERE o.status <> 'CANCELLED' AND d.product_code IN ({placeholders})
        GROUP BY d.product_code
    """, checked)
    sold = {code: int(qty) for code, qty in cur.fetchall()}
    cur.execute("SELECT status, COUNT(*) FROM orders GROUP BY status")
    orders_by_status = dict(cur.fetchall())
conn.close()

print("Orders by status:", orders_by_status)
all_consistent = True
for code in checked:
    remaining, sold_qty, initial = stock[code], sold.get(code, 0), initial_stock[code]
    consistent = remaining >= 0 and remaining + sold_qty == initial
    all_consistent &= consistent
    label = "hot" if code == bench_config["hot_product"] else "multi"
    print(f"Product {code} ({label}): initial={initial}, remaining={remaining}, sold={sold_qty}, consistent={consistent}")

print("STOCK CONSISTENT" if all_consistent else "STOCK INCONSISTENT")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

metrics_df = pd.DataFrame(system_metrics)
if not metrics_df.empty:
    metrics_df["elapsed_sec"] = metrics_df["timestamp"] - metrics_df["timestamp"].min()

stats_file = "/kaggle/working/kaggle_load_results_stats.csv"
if os.path.exists(stats_file):
    stats_df = pd.read_csv(stats_file)
    display_cols = [c for c in ["Type", "Name", "Request Count", "Failure Count", "Median Response Time", "95%", "Requests/s"] if c in stats_df.columns]
    print("Locust Endpoint Statistics:")
    display(stats_df[display_cols])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

# CPU and RAM usage
if not metrics_df.empty:
    ax1.plot(metrics_df["elapsed_sec"], metrics_df["cpu_total_pct"], label="CPU Total (%)", color="#1f77b4", linewidth=1.8)
    ax1.axhline(100, color="red", linestyle="--", alpha=0.5, label="100% CPU")
    ax1.set_title("System Resource Utilization (CPU & RAM)")
    ax1.set_xlabel("Elapsed Time (s)")
    ax1.set_ylabel("CPU (%)", color="#1f77b4")
    ax1.set_ylim(0, 105)
    ax1.grid(True, linestyle="--", alpha=0.4)

    ax1_ram = ax1.twinx()
    ax1_ram.plot(metrics_df["elapsed_sec"], metrics_df["ram_used_mb"], label="RAM (MB)", color="#9467bd", linestyle=":", linewidth=1.8)
    ax1_ram.set_ylabel("RAM Used (MB)", color="#9467bd")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1_ram.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

# Throughput vs p95 Latency
history_file = "/kaggle/working/kaggle_load_results_stats_history.csv"
if os.path.exists(history_file) and os.path.getsize(history_file) > 0:
    hist_df = pd.read_csv(history_file)

    if "Name" in hist_df.columns and "Aggregated" in hist_df["Name"].values:
        hist_df = hist_df[hist_df["Name"] == "Aggregated"].copy()

    time_col = "Timestamp" if "Timestamp" in hist_df.columns else hist_df.columns[0]
    hist_df["elapsed_sec"] = hist_df[time_col] - hist_df[time_col].min()

    rps_col = next((c for c in ["Requests/s", "Total Requests/s", "Current RPS", "Requests/sec"] if c in hist_df.columns), None)
    p95_col = next((c for c in ["95%", "95%tile", "Total 95%"] if c in hist_df.columns), None)

    if rps_col:
        ax2.plot(hist_df["elapsed_sec"], hist_df[rps_col], color="#2ca02c", label=f"RPS ({rps_col})", linewidth=1.8)
        ax2.set_ylabel("Requests / sec", color="#2ca02c")

    if p95_col:
        ax2_twin = ax2.twinx()
        ax2_twin.plot(hist_df["elapsed_sec"], hist_df[p95_col], color="#ff7f0e", linestyle="--", label="p95 Latency (ms)", linewidth=1.8)
        ax2_twin.set_ylabel("Latency (ms)", color="#ff7f0e")

    ax2.set_title("Throughput (RPS) vs p95 Latency (ms)")
    ax2.set_xlabel("Elapsed Time (s)")
    ax2.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/kaggle/working/kaggle_load_benchmark.png", dpi=150)
plt.show()
print("Report chart saved to /kaggle/working/kaggle_load_benchmark.png")